# PRISM Dataset EDA

**Run on:** EC2 after syncing 2-4 sessions from S3. NOT the full 4 TB.

**Real data structure (confirmed from S3):**
```
{session}/
  sequence_001/
    rgb/                  *.png  (timestamp stems e.g. 1767667361_605)
    polar/
      0d/  45d/  90d/  135d/    *.png  (individual angle images)
    lidar_accum_scan/     *.pcd
  sequence_002/ ...
  vehicle_state/
```

**Timestamp format:** `{unix_seconds}_{milliseconds}` e.g. `1767667361_605` = 1767667361.605 seconds

**Questions this notebook answers:**
1. Actual pixel value range — uint8 (0-255) or uint16 (0-65535)?
2. What do RGB vs polar images look like side by side?
3. Does DoLP visually differ between dry / wet / snow conditions?
4. Class distribution — how imbalanced are surface states?
5. Per-channel Stokes statistics needed for normalization in transforms.py

In [ ]:
import os, sys, json, random
sys.path.insert(0, os.path.abspath(".."))

# ── CONFIG (override via env vars when running headlessly) ───────────────────
DATA_ROOT     = os.environ.get("POLARIS_DATA_ROOT",  "/home/ubuntu/polaris_data")
LABELS_JSON   = os.environ.get("POLARIS_LABELS_JSON", f"{DATA_ROOT}/labels.json")
SAMPLE_N      = 50       # frames per session for per-channel stats (cell 8)
DISPLAY_SCALE = 4        # downsample factor for display
# ─────────────────────────────────────────────────────────────────────────────

import matplotlib
matplotlib.use("Agg")   # headless — no display needed

from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tqdm import tqdm

from src.data.stokes import compute_stokes

plt.rcParams["figure.dpi"] = 110

root    = Path(DATA_ROOT)
labels  = json.load(open(LABELS_JSON))
folders = labels["folders"]

all_sessions = []
for split in ["train", "val"]:
    split_dir = root / split
    if split_dir.exists():
        all_sessions += [(split, d.name, d) for d in sorted(split_dir.iterdir()) if d.is_dir()]

print(f"Sessions found: {len(all_sessions)}")
for split, name, _ in all_sessions:
    print(f"  [{split}] {name}")

## 1. Directory Structure — verify layout of one session

In [ ]:
_, session_name, session_dir = all_sessions[0]
print(f"Session: {session_name}")
for p in sorted(session_dir.rglob("*"))[:50]:
    rel = p.relative_to(session_dir)
    prefix = "  " * (len(rel.parts) - 1)
    print(f"{prefix}{rel.name}{'/' if p.is_dir() else ''}")

## 2. Pixel Value Range — confirm 12-bit packing

In [ ]:
from src.data.utils import _parse_frame_ts

_, session_name, session_dir = all_sessions[0]
seq_dir = next(d for d in sorted(session_dir.iterdir()) if d.is_dir() and d.name != "vehicle_state")

# Pick one frame stem
rgb_frames = sorted((seq_dir / "rgb").glob("*.png"))
stem = rgb_frames[10].stem
print(f"Sample frame stem : {stem}")
print(f"As seconds        : {_parse_frame_ts(stem):.3f}\n")

for angle, subdir in [("0","0d"),("45","45d"),("90","90d"),("135","135d")]:
    p = seq_dir / "polar" / subdir / f"{stem}.png"
    arr = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
    print(f"  Polar {angle:>3}° | dtype={arr.dtype} | min={arr.min():6d} | max={arr.max():6d} | shape={arr.shape}")

rgb_arr = cv2.imread(str(seq_dir / "rgb" / f"{stem}.png"), cv2.IMREAD_UNCHANGED)
print(f"  RGB          | dtype={rgb_arr.dtype} | min={rgb_arr.min():6d} | max={rgb_arr.max():6d} | shape={rgb_arr.shape}")

# Determine bit depth from actual max value
max_val = int(rgb_arr.max())
if max_val <= 255:
    print("\nuint8 image  → divide by 255.0 during loading")
elif max_val <= 4095:
    print("\n12-bit in uint16 → divide by 4095.0 during loading")
else:
    print("\n16-bit image → divide by 65535.0 during loading")

## 3. Visual: RGB vs 4 Polar Angles — same frame

In [ ]:
sc = DISPLAY_SCALE

# Auto-detect max value from one image
_probe = cv2.imread(str(seq_dir / "rgb" / f"{stem}.png"), cv2.IMREAD_UNCHANGED)
_max = int(_probe.max())
MAX_VAL = 255 if _max <= 255 else (4095 if _max <= 4095 else 65535)
print(f"Detected MAX_VAL={MAX_VAL}  (image max pixel={_max})")

def load_norm(path):
    arr = cv2.imread(str(path), cv2.IMREAD_UNCHANGED).astype(np.float32)
    return np.clip(arr / MAX_VAL, 0, 1)

def load_raw(path):
    return cv2.imread(str(path), cv2.IMREAD_UNCHANGED).astype(np.float32)

rgb  = cv2.cvtColor(cv2.imread(str(seq_dir/"rgb"/f"{stem}.png"), cv2.IMREAD_UNCHANGED), cv2.COLOR_BGR2RGB)
rgb  = np.clip(rgb.astype(np.float32) / MAX_VAL, 0, 1)
p0   = load_norm(seq_dir/"polar"/"0d"/f"{stem}.png")
p45  = load_norm(seq_dir/"polar"/"45d"/f"{stem}.png")
p90  = load_norm(seq_dir/"polar"/"90d"/f"{stem}.png")
p135 = load_norm(seq_dir/"polar"/"135d"/f"{stem}.png")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, title, img in zip(axes.flat,
    ["RGB", "Polar 0°", "Polar 45°", "Polar 90°", "Polar 135°", ""],
    [rgb, p0, p45, p90, p135, None]):
    if img is None: ax.axis("off"); continue
    disp = img[::sc, ::sc] if img.ndim == 2 else img[::sc, ::sc, :]
    cmap = None if img.ndim == 3 else "gray"
    ax.imshow(disp, cmap=cmap, vmin=0, vmax=1)
    ax.set_title(title, fontsize=13); ax.axis("off")

plt.suptitle(f"{session_name} / {seq_dir.name} / frame {stem}", fontsize=10)
plt.tight_layout(); plt.show()

## 4. Stokes Parameters and DoLP Heatmap

In [ ]:
i0   = load_raw(seq_dir/"polar"/"0d"/f"{stem}.png")
i45  = load_raw(seq_dir/"polar"/"45d"/f"{stem}.png")
i90  = load_raw(seq_dir/"polar"/"90d"/f"{stem}.png")
i135 = load_raw(seq_dir/"polar"/"135d"/f"{stem}.png")

stokes = compute_stokes(i0, i45, i90, i135, eps=1.0)

print("Stokes channel statistics:")
for name, arr in stokes.items():
    print(f"  {name:5s} | min={arr.min():.4f}  max={arr.max():.4f}  mean={arr.mean():.4f}  std={arr.std():.4f}")

fig, axes = plt.subplots(1, 5, figsize=(22, 5))
cmaps = {"S0":"gray", "S1":"RdBu_r", "S2":"RdBu_r", "DoLP":"hot", "AoLP":"hsv"}
for ax, (name, arr) in zip(axes, stokes.items()):
    im = ax.imshow(arr[::sc, ::sc], cmap=cmaps[name])
    ax.set_title(name, fontsize=13); ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle("High DoLP on road = wet/icy surface (Fresnel reflection)", fontsize=11)
plt.tight_layout(); plt.show()

## 5. Label Lookup — verify for sample frames

In [ ]:
from src.data.utils import lookup_label, _parse_frame_ts as _ns_to_sec

_, session_name, session_dir = all_sessions[0]
folder_entry = folders.get(session_name, {})

print(f"Session: {session_name}")
print(f"Label structure keys: {list(folder_entry.keys())}\n")

# Test lookup on 5 frames
seq_dir = next(d for d in sorted(session_dir.iterdir()) if d.is_dir() and d.name != "vehicle_state")
frames = sorted((seq_dir / "rgb").glob("*.png"))[:5]

print(f"{'Stem (ns)':>25}  {'Seconds':>16}  Label")
print("-" * 75)
for f in frames:
    stem = f.stem
    ts_sec = _ns_to_sec(stem)
    label = lookup_label(folder_entry, ts_sec)
    print(f"  {stem:>25}  {ts_sec:>16.3f}  {label}")

## 6. Class Distribution Across All Available Sessions

In [ ]:
records = []

for split, session_name, session_dir in tqdm(all_sessions, desc="indexing"):
    folder_entry = folders.get(session_name)
    if not folder_entry:
        continue
    for seq_dir in sorted(session_dir.iterdir()):
        if not seq_dir.is_dir() or seq_dir.name == "vehicle_state":
            continue
        rgb_dir = seq_dir / "rgb"
        if not rgb_dir.exists():
            continue
        for f in rgb_dir.glob("*.png"):
            ts_sec = _ns_to_sec(f.stem)
            label = lookup_label(folder_entry, ts_sec)
            if label:
                records.append({
                    "split":            split,
                    "session":          session_name,
                    "surface_state":    label.get("surface_state"),
                    "surface_material": label.get("surface_material"),
                    "weather":          label.get("weather"),
                    "road_type":        label.get("road_type"),
                })

df = pd.DataFrame(records)
print(f"Total labeled frames: {len(df)}")
print(f"Excluded (transitions): {sum(1 for _ in records) - len(df)}")
print(df.head())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, col in zip(axes.flat, ["surface_state", "surface_material", "weather", "road_type"]):
    counts = df[col].value_counts()
    counts.plot.bar(ax=ax, color="steelblue", edgecolor="white")
    ax.set_title(col.replace("_", " ").title(), fontsize=13)
    ax.set_xlabel("")
    for bar, count in zip(ax.patches, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f"{count:,}", ha='center', va='bottom', fontsize=9)

plt.suptitle("PRISM Class Distribution (sampled sessions)", fontsize=13)
plt.tight_layout(); plt.show()

print("\nSurface state distribution:")
print(df["surface_state"].value_counts().to_string())
print("\nSurface material distribution:")
print(df["surface_material"].value_counts().to_string())

## 7. DoLP Comparison Across Conditions

In [ ]:
# Pick one session per available condition for visual comparison
# Assign sessions manually here after seeing cell 6 distribution

CONDITION_MAP = {}  # e.g. {"dry": all_sessions[0], "wet": all_sessions[1]}
for split, name, sdir in all_sessions:
    fe = folders.get(name, {})
    segs = fe.get("ordered_segments", [])
    default = fe.get("default_label", {})
    state = (segs[0]["label"].get("surface_state") if segs
             else default.get("surface_state", "unknown"))
    if state not in CONDITION_MAP:
        CONDITION_MAP[state] = (split, name, sdir)

print("Condition → session mapping:")
for cond, (split, name, _) in CONDITION_MAP.items():
    print(f"  {cond:15s} → [{split}] {name}")

fig, axes = plt.subplots(len(CONDITION_MAP), 2, figsize=(12, 4 * len(CONDITION_MAP)))
if len(CONDITION_MAP) == 1: axes = [axes]

for row, (cond, (split, name, sdir)) in enumerate(CONDITION_MAP.items()):
    seq = next(d for d in sorted(sdir.iterdir()) if d.is_dir() and d.name != "vehicle_state")
    s = sorted((seq/"rgb").glob("*.png"))[5].stem

    rgb_img = cv2.cvtColor(cv2.imread(str(seq/"rgb"/f"{s}.png"), cv2.IMREAD_UNCHANGED), cv2.COLOR_BGR2RGB)
    rgb_img = np.clip(rgb_img.astype(np.float32) / MAX_VAL, 0, 1)

    i0_  = load_raw(seq/"polar"/"0d"/f"{s}.png")
    i45_ = load_raw(seq/"polar"/"45d"/f"{s}.png")
    i90_ = load_raw(seq/"polar"/"90d"/f"{s}.png")
    i135_= load_raw(seq/"polar"/"135d"/f"{s}.png")
    st   = compute_stokes(i0_, i45_, i90_, i135_)

    axes[row][0].imshow(rgb_img[::sc,::sc,:])
    axes[row][0].set_title(f"{cond.upper()} — RGB"); axes[row][0].axis("off")
    im = axes[row][1].imshow(st["DoLP"][::sc,::sc], cmap="hot", vmin=0, vmax=0.5)
    axes[row][1].set_title(f"{cond.upper()} — DoLP (mean={st['DoLP'].mean():.3f})")
    axes[row][1].axis("off"); plt.colorbar(im, ax=axes[row][1])

plt.tight_layout(); plt.show()

## 8. Per-Channel Stokes Statistics (for transforms.py normalization)

In [ ]:
channel_names = ["S0", "S1", "S2", "DoLP", "AoLP"]
all_means = {c: [] for c in channel_names}
all_stds  = {c: [] for c in channel_names}

for split, session_name, session_dir in tqdm(all_sessions):
    for seq_dir in sorted(session_dir.iterdir()):
        if not seq_dir.is_dir() or seq_dir.name == "vehicle_state":
            continue
        frames = sorted((seq_dir/"polar"/"0d").glob("*.png"))
        sampled = random.sample(frames, min(SAMPLE_N, len(frames)))
        for f in sampled:
            stem = f.stem
            try:
                i0_  = load_raw(seq_dir/"polar"/"0d"/f"{stem}.png")
                i45_ = load_raw(seq_dir/"polar"/"45d"/f"{stem}.png")
                i90_ = load_raw(seq_dir/"polar"/"90d"/f"{stem}.png")
                i135_= load_raw(seq_dir/"polar"/"135d"/f"{stem}.png")
                st = compute_stokes(i0_, i45_, i90_, i135_)
                for ch in channel_names:
                    all_means[ch].append(float(st[ch].mean()))
                    all_stds[ch].append(float(st[ch].std()))
            except Exception as e:
                pass

print("\nPolar channel statistics — copy these into src/data/transforms.py:\n")
polar_mean, polar_std = [], []
for ch in channel_names:
    m = float(np.mean(all_means[ch]))
    s = float(np.mean(all_stds[ch]))
    polar_mean.append(round(m, 6))
    polar_std.append(round(s, 6))
    print(f"  {ch:5s}  mean={m:.4f}  std={s:.4f}")

print(f"\npolar_mean = {polar_mean}")
print(f"polar_std  = {polar_std}")

## Summary — What to Update Before Training

| Finding | File to update |
|---|---|
| Pixel max value (4095 or 65535) | [src/data/dataset.py](../src/data/dataset.py) — `_load_rgb()` division constant |
| `polar_mean` / `polar_std` values | [src/data/transforms.py](../src/data/transforms.py) |
| Class imbalance (rare: snow, slush, gravel) | [src/training/trainer.py](../src/training/trainer.py) — add `class_weight` to CrossEntropyLoss |

## 9. Polar vs RGB Resolution — confirm super-pixel ratio

In [ ]:
_, session_name, session_dir = all_sessions[0]
seq_dir = next(d for d in sorted(session_dir.iterdir()) if d.is_dir() and d.name != "vehicle_state")
frames  = sorted((seq_dir / "rgb").glob("*.png"))
stem    = frames[10].stem

rgb_img  = cv2.imread(str(seq_dir / "rgb"           / f"{stem}.png"), cv2.IMREAD_UNCHANGED)
p0_img   = cv2.imread(str(seq_dir / "polar" / "0d"  / f"{stem}.png"), cv2.IMREAD_UNCHANGED)
flat_img = cv2.imread(str(seq_dir / "polar"          / f"{stem}.png"), cv2.IMREAD_UNCHANGED)

print(f"RGB shape          : {rgb_img.shape}   dtype={rgb_img.dtype}")
print(f"Polar/0d shape     : {p0_img.shape}   dtype={p0_img.dtype}")
if flat_img is not None:
    print(f"Polar/root shape   : {flat_img.shape}   dtype={flat_img.dtype}  channels={flat_img.shape[2] if flat_img.ndim==3 else 1}")
else:
    print("Polar/root file    : not found for this frame")

if rgb_img is not None and p0_img is not None:
    h_ratio = rgb_img.shape[0] / p0_img.shape[0]
    w_ratio = rgb_img.shape[1] / p0_img.shape[1]
    print(f"\nResolution ratio   : {h_ratio:.1f}x height, {w_ratio:.1f}x width")
    if abs(h_ratio - 2.0) < 0.1:
        print("Confirmed 2x2 super-pixel sensor (polar is half RGB resolution)")
    elif abs(h_ratio - 1.0) < 0.1:
        print("Polar and RGB are SAME resolution — no super-pixel demosaicing")
    else:
        print(f"Unexpected ratio — check sensor docs")

## 10. Missing Frame Audit — check timestamp alignment across modalities

In [ ]:
from src.data.utils import _parse_frame_ts

missing_polar  = []
missing_rgb    = []
missing_angles = []
total_frames   = 0

for split, session_name, session_dir in all_sessions:
    for seq_dir in sorted(session_dir.iterdir()):
        if not seq_dir.is_dir() or seq_dir.name == "vehicle_state":
            continue

        rgb_stems   = {f.stem for f in (seq_dir / "rgb").glob("*.png")} if (seq_dir / "rgb").exists() else set()
        polar_stems = {f.stem for f in (seq_dir / "polar" / "0d").glob("*.png")} if (seq_dir / "polar" / "0d").exists() else set()

        total_frames += len(rgb_stems)

        for stem in rgb_stems - polar_stems:
            missing_polar.append(f"{session_name}/{seq_dir.name}/{stem}")

        for stem in polar_stems - rgb_stems:
            missing_rgb.append(f"{session_name}/{seq_dir.name}/{stem}")

        # Check all 4 angles present for polar frames
        for stem in polar_stems:
            for angle_dir in ["0d", "45d", "90d", "135d"]:
                if not (seq_dir / "polar" / angle_dir / f"{stem}.png").exists():
                    missing_angles.append(f"{session_name}/{seq_dir.name}/{stem} — missing {angle_dir}")

print(f"Total RGB frames       : {total_frames:,}")
print(f"Missing polar (in RGB not polar): {len(missing_polar)}")
print(f"Missing RGB   (in polar not RGB): {len(missing_rgb)}")
print(f"Incomplete polar angles         : {len(missing_angles)}")

if missing_polar[:5]:
    print(f"\nSample missing polar: {missing_polar[:5]}")
if missing_angles[:5]:
    print(f"\nSample incomplete angles: {missing_angles[:5]}")

if len(missing_polar) == 0 and len(missing_angles) == 0:
    print("\nAll timestamps aligned — dataset is clean")

## 11. DoLP Histogram — does polarimetry separate surface states?
This is the core scientific validation of POLARIS.

In [ ]:
from src.data.utils import lookup_label

dolp_by_state = {}   # {surface_state: [mean_dolp_values]}
rgb_mean_by_state = {}

for split, session_name, session_dir in tqdm(all_sessions, desc="computing DoLP"):
    folder_entry = folders.get(session_name)
    if not folder_entry:
        continue

    for seq_dir in sorted(session_dir.iterdir()):
        if not seq_dir.is_dir() or seq_dir.name == "vehicle_state":
            continue

        frames = sorted((seq_dir / "polar" / "0d").glob("*.png"))
        sampled = random.sample(frames, min(30, len(frames)))

        for f in sampled:
            stem   = f.stem
            ts_sec = _parse_frame_ts(stem)
            label  = lookup_label(folder_entry, ts_sec)
            if not label:
                continue
            state = label.get("surface_state", "unknown")

            try:
                i0   = cv2.imread(str(seq_dir/"polar"/"0d" /f"{stem}.png"), cv2.IMREAD_UNCHANGED).astype(np.float32)
                i45  = cv2.imread(str(seq_dir/"polar"/"45d"/f"{stem}.png"), cv2.IMREAD_UNCHANGED).astype(np.float32)
                i90  = cv2.imread(str(seq_dir/"polar"/"90d"/f"{stem}.png"), cv2.IMREAD_UNCHANGED).astype(np.float32)
                i135 = cv2.imread(str(seq_dir/"polar"/"135d"/f"{stem}.png"), cv2.IMREAD_UNCHANGED).astype(np.float32)
                st   = compute_stokes(i0, i45, i90, i135)
                dolp_by_state.setdefault(state, []).append(float(st["DoLP"].mean()))

                rgb = cv2.imread(str(seq_dir/"rgb"/f"{stem}.png"), cv2.IMREAD_UNCHANGED)
                if rgb is not None:
                    rgb_mean_by_state.setdefault(state, []).append(float(rgb.mean()) / MAX_VAL)
            except Exception:
                pass

# DoLP histogram overlay
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
colors = {"dry":"steelblue","damp":"mediumseagreen","wet":"coral","slush":"orchid","snow_covered":"gold"}

for state, vals in sorted(dolp_by_state.items()):
    axes[0].hist(vals, bins=40, alpha=0.65, label=f"{state} (n={len(vals)})",
                 color=colors.get(state, "grey"), density=True)
axes[0].set_xlabel("Mean DoLP per frame"); axes[0].set_ylabel("Density")
axes[0].set_title("DoLP Distribution by Surface State\n(higher = more polarised = wet/icy)")
axes[0].legend(); axes[0].grid(alpha=0.3)

# RGB mean overlay
for state, vals in sorted(rgb_mean_by_state.items()):
    axes[1].hist(vals, bins=40, alpha=0.65, label=f"{state} (n={len(vals)})",
                 color=colors.get(state, "grey"), density=True)
axes[1].set_xlabel("Mean RGB brightness per frame"); axes[1].set_ylabel("Density")
axes[1].set_title("RGB Brightness Distribution by Surface State\n(does RGB alone separate conditions?)")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.savefig("dolp_separation.png", dpi=150); plt.show()

print("\nMean DoLP per surface state:")
for state, vals in sorted(dolp_by_state.items()):
    print(f"  {state:15s}: {np.mean(vals):.4f}  ± {np.std(vals):.4f}  (n={len(vals)})")

## 12. Stokes Channel Correlation Matrix — are any channels redundant?

In [ ]:
channel_names = ["S0", "S1", "S2", "DoLP", "AoLP"]
channel_data  = {c: [] for c in channel_names}

_, _, session_dir = all_sessions[0]
seq_dir = next(d for d in sorted(session_dir.iterdir()) if d.is_dir() and d.name != "vehicle_state")
frames  = sorted((seq_dir / "polar" / "0d").glob("*.png"))
sampled = random.sample(frames, min(60, len(frames)))

for f in tqdm(sampled, desc="correlation sample"):
    stem = f.stem
    try:
        i0   = cv2.imread(str(seq_dir/"polar"/"0d" /f"{stem}.png"), cv2.IMREAD_UNCHANGED).astype(np.float32)
        i45  = cv2.imread(str(seq_dir/"polar"/"45d"/f"{stem}.png"), cv2.IMREAD_UNCHANGED).astype(np.float32)
        i90  = cv2.imread(str(seq_dir/"polar"/"90d"/f"{stem}.png"), cv2.IMREAD_UNCHANGED).astype(np.float32)
        i135 = cv2.imread(str(seq_dir/"polar"/"135d"/f"{stem}.png"), cv2.IMREAD_UNCHANGED).astype(np.float32)
        st   = compute_stokes(i0, i45, i90, i135)
        for ch in channel_names:
            channel_data[ch].append(float(st[ch].mean()))
    except Exception:
        pass

corr_df = pd.DataFrame(channel_data).corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(corr_df, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            vmin=-1, vmax=1, ax=axes[0], square=True)
axes[0].set_title("Stokes Channel Correlation\n(|r|>0.9 = redundant channel)")

# Also show per-channel variance — low variance = not useful
variances = {ch: np.var(channel_data[ch]) for ch in channel_names}
axes[1].bar(variances.keys(), variances.values(), color="steelblue", edgecolor="white")
axes[1].set_title("Per-Channel Variance\n(low = channel carries little information)")
axes[1].set_ylabel("Variance of per-frame means")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout(); plt.savefig("stokes_correlation.png", dpi=150); plt.show()
print("\nCorrelation matrix:")
print(corr_df.round(3).to_string())

## 13. Class Co-occurrence — weather × surface state heatmap

In [ ]:
# df is built in cell 6 — reuse it
if "df" not in dir() or df is None or len(df) == 0:
    print("Run cell 6 (Class Distribution) first to build df")
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Weather × Surface State
    ct1 = pd.crosstab(df["weather"], df["surface_state"])
    sns.heatmap(ct1, annot=True, fmt="d", cmap="YlOrRd", ax=axes[0])
    axes[0].set_title("Weather × Surface State\n(frame counts)")

    # Surface Material × Surface State
    ct2 = pd.crosstab(df["surface_material"], df["surface_state"])
    sns.heatmap(ct2, annot=True, fmt="d", cmap="YlOrRd", ax=axes[1])
    axes[1].set_title("Surface Material × Surface State\n(frame counts)")

    plt.tight_layout(); plt.savefig("class_cooccurrence.png", dpi=150); plt.show()

    # Rare class warning
    print("\nClass balance check:")
    state_counts = df["surface_state"].value_counts()
    total = len(df)
    for state, count in state_counts.items():
        pct = count / total * 100
        flag = " ← RARE, consider class weights" if pct < 10 else ""
        print(f"  {state:15s}: {count:5,} frames ({pct:.1f}%){flag}")

## 14. Final Findings Summary — copy these values into training config

In [ ]:
print("=" * 60)
print("POLARIS EDA — FINDINGS SUMMARY")
print("=" * 60)

print(f"\n1. IMAGE FORMAT")
print(f"   MAX_VAL     = {MAX_VAL}  → divide by this in _load_rgb()")
print(f"   RGB dtype   = {rgb_img.dtype}")
print(f"   RGB shape   = {rgb_img.shape}")
print(f"   Polar shape = {p0_img.shape}")

print(f"\n2. POLAR STATS — update src/data/transforms.py")
print(f"   polar_mean = {polar_mean}")
print(f"   polar_std  = {polar_std}")

print(f"\n3. DOLP SEPARATION")
for state, vals in sorted(dolp_by_state.items()):
    print(f"   {state:15s}: mean DoLP = {np.mean(vals):.4f}")

print(f"\n4. CLASS BALANCE")
if "df" in dir() and df is not None and len(df) > 0:
    state_counts = df["surface_state"].value_counts()
    total = len(df)
    for state, count in state_counts.items():
        pct = count / total * 100
        flag = " ← add class weight" if pct < 10 else ""
        print(f"   {state:15s}: {pct:.1f}%{flag}")

print(f"\n5. MISSING FRAMES")
print(f"   Missing polar angles: {len(missing_angles)}")
print(f"   Misaligned timestamps: {len(missing_polar) + len(missing_rgb)}")

print("\n" + "=" * 60)
print("Next steps:")
print("  1. Update MAX_VAL divisor in src/data/dataset.py _load_rgb()")
print("  2. Update polar_mean/polar_std in src/data/transforms.py")
print("  3. Add class weights if any surface state < 10%")
print("  4. Run precompute_stokes.py on full dataset")
print("  5. Launch training on g5.xlarge")